<!-- NOTEBOOK_METADATA title: "Observability for Streamlit with Langfuse" sidebarTitle: "Streamlit" logo: "/images/integrations/streamlit_icon.svg" description: "Build an LLM chat UI with Streamlit and trace it with Langfuse." category: "Integrations" -->

# Build an LLM Chat UI with Streamlit and trace it with Langfuse

This is a simple end-to-end example that shows how to integrate a [Streamlit](https://streamlit.io/) application with [Langfuse](https://langfuse.com) for LLM Observability and Evaluation.

**Note:** Unlike a typical notebook guide, Streamlit apps are run as scripts with `streamlit run app.py`. The code blocks in this notebook compose into one `app.py` — the full, runnable version is available in [`examples/streamlit-demo/app.py`](https://github.com/langfuse/langfuse-docs/tree/main/examples/streamlit-demo).

## Introduction

### What is Streamlit?

[Streamlit](https://streamlit.io/) is an open-source Python framework for quickly building interactive data and AI apps. It turns regular Python scripts into web UIs without requiring any frontend code. See the [docs](https://docs.streamlit.io/) for more details.

### What is Langfuse?

[Langfuse](https://github.com/langfuse/langfuse) is an open-source LLM engineering platform that helps build reliable LLM applications via LLM Application Observability, Evaluation, Experiments, and Prompt Management. See the [docs](https://langfuse.com/docs) for more details.

### Outline

This notebook will show you how to

1. Build a streaming chat interface in Python using [Streamlit’s chat elements](https://docs.streamlit.io/develop/api-reference/chat)
2. Add [Langfuse Tracing](https://langfuse.com/docs/tracing) to the chatbot
3. Implement additional Langfuse features used frequently in chat applications: [chat sessions](https://langfuse.com/docs/tracing-features/sessions), [user identification](https://langfuse.com/docs/tracing-features/users), [user feedback](https://langfuse.com/docs/scores/user-feedback), and routing across multiple [model providers](https://langfuse.com/integrations/model-providers)

## Setup

Install requirements. We use the OpenAI and Anthropic SDKs for this example to show Langfuse’s cross-provider instrumentation in a single app.

In [ ]:
%pip install streamlit "langfuse>=4.0.0" openai anthropic opentelemetry-instrumentation-anthropic python-dotenv

Set credentials and point the Langfuse SDK at your project. You can either create a free [Langfuse Cloud](https://cloud.langfuse.com) account or [self-host Langfuse](https://langfuse.com/self-hosting) in a couple of minutes.

In [ ]:
import os

# Get keys for your project from the project settings page
# https://cloud.langfuse.com
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"  # 🇪🇺 EU region
# os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"  # 🇺🇸 US region

os.environ["OPENAI_API_KEY"] = "sk-..."
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

Initialize the Langfuse client and enable auto-instrumentation. `langfuse.openai.OpenAI` is a drop-in replacement that traces every OpenAI call. For Anthropic we enable the OpenTelemetry instrumentor — this keeps the Anthropic SDK itself untouched while still auto-capturing every call.

In [ ]:
import atexit
import streamlit as st
from anthropic import Anthropic
from langfuse import get_client
from langfuse.openai import OpenAI
from opentelemetry.instrumentation.anthropic import AnthropicInstrumentor


@st.cache_resource
def init_langfuse():
    client = get_client()
    if client.auth_check():
        print("Langfuse client authenticated")
    else:
        print("Langfuse authentication failed — check LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_BASE_URL")
    atexit.register(client.flush)
    return client


@st.cache_resource
def init_anthropic_instrumentor():
    AnthropicInstrumentor().instrument()


langfuse = init_langfuse()
init_anthropic_instrumentor()
openai_client = OpenAI()
anthropic_client = Anthropic()

## Implementation of Chat functions

### Sessions

Each Streamlit browser tab has its own [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state), which persists across reruns but resets when the tab is closed. We mint a UUID the first time the tab loads and reuse it for every trace produced by the same conversation — this becomes the [Langfuse Session](https://langfuse.com/docs/tracing-features/sessions) ID.

In [ ]:
import uuid

if "messages" not in st.session_state:
    st.session_state.messages = []

if "session_id" not in st.session_state:
    st.session_state.session_id = str(uuid.uuid4())

### User identification

A sidebar text input lets the user claim a name. The value is stored in session state and later attached to traces via [`propagate_attributes(user_id=...)`](https://langfuse.com/docs/tracing-features/users). If no name is provided we fall back to `"anonymous"` so every trace still carries a user id.

In [ ]:
if "user_id" not in st.session_state:
    st.session_state.user_id = "anonymous"

with st.sidebar:
    name = st.text_input(
        "Your name",
        value="" if st.session_state.user_id == "anonymous" else st.session_state.user_id,
    )
    st.session_state.user_id = name.strip() or "anonymous"

### Response handler

We wrap the model call in the Langfuse [`@observe()` decorator](https://langfuse.com/docs/sdk/python/decorators) so each assistant reply becomes a trace. Inside the decorator we open a [`propagate_attributes`](https://langfuse.com/docs/tracing-features/sessions) context that attaches `session_id`, `user_id`, and a provider tag to every span produced by the SDK calls below it.

We also capture the current `trace_id` into a caller-supplied list so the UI can reference it later when the user clicks 👍 or 👎.

In [ ]:
from langfuse import observe, propagate_attributes


def _stream_openai(messages, model):
    stream = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        stream_options={"include_usage": True},
    )
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            yield delta


def _stream_anthropic(messages, model):
    stream = anthropic_client.messages.create(
        model=model,
        max_tokens=1024,
        messages=messages,
        stream=True,
    )
    for event in stream:
        if event.type == "content_block_delta":
            yield event.delta.text


@observe()
def stream_reply(messages, model, trace_holder):
    trace_holder.append(langfuse.get_current_trace_id())
    # The UI stores an extra `trace_id` on each assistant message for feedback scoring.
    # OpenAI tolerates unknown keys but Anthropic rejects them, so we project to {role, content}.
    clean = [{"role": m["role"], "content": m["content"]} for m in messages]
    if MODELS[model] == "openai":
        yield from _stream_openai(clean, model)
    else:
        yield from _stream_anthropic(clean, model)

### Multi-provider support

A `MODELS` dict maps each selectable model to its provider, and a sidebar dropdown lets the user switch at any time. Both providers are traced automatically: OpenAI via the `langfuse.openai` wrapper, Anthropic via the OpenTelemetry instrumentor we enabled during setup. We also pass the provider name as a Langfuse [tag](https://langfuse.com/docs/tracing-features/tags) so filtering by provider in the UI is one click.

In [ ]:
MODELS = {
    "gpt-4o-mini": "openai",
    "claude-sonnet-4-6": "anthropic",
}

if "model" not in st.session_state:
    st.session_state.model = "gpt-4o-mini"

with st.sidebar:
    st.session_state.model = st.selectbox(
        "Model",
        list(MODELS.keys()),
        index=list(MODELS.keys()).index(st.session_state.model),
    )

### User feedback handler

We implement [user feedback tracking](https://langfuse.com/docs/scores/user-feedback) as [`CATEGORICAL` scores](https://langfuse.com/docs/scores/data-types) attached to the trace id captured in the response handler. `👍` and `👎` buttons appear next to each assistant message; after a user clicks, we disable the buttons for that message by tracking scored trace ids in session state.

In [ ]:
if "scored_traces" not in st.session_state:
    st.session_state.scored_traces = set()


def score_trace(trace_id: str, value: str):
    langfuse.create_score(
        trace_id=trace_id,
        name="user-feedback",
        value=value,
        data_type="CATEGORICAL",
    )
    langfuse.flush()
    st.session_state.scored_traces.add(trace_id)

## Run Streamlit App

The snippets above combine into a single `app.py`. Start it with:

```bash
streamlit run app.py
```

Streamlit opens the chat UI at `http://localhost:8501`. A new session id is minted per browser tab, and the sidebar lets you set your name, switch models, and start a fresh conversation.

Here is the complete `app.py` for reference — it matches [`examples/streamlit-demo/app.py`](https://github.com/langfuse/langfuse-docs/tree/main/examples/streamlit-demo) in the Langfuse docs repo:

In [ ]:
import atexit
import uuid

from anthropic import Anthropic
from dotenv import load_dotenv
from langfuse import get_client, observe, propagate_attributes
from langfuse.openai import OpenAI
from opentelemetry.instrumentation.anthropic import AnthropicInstrumentor
import streamlit as st

load_dotenv()

MODELS = {
    "gpt-4o-mini": "openai",
    "claude-sonnet-4-6": "anthropic",
}


@st.cache_resource
def init_langfuse():
    client = get_client()
    if client.auth_check():
        print("Langfuse client authenticated")
    else:
        print("Langfuse authentication failed — check LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_BASE_URL")
    atexit.register(client.flush)
    return client


@st.cache_resource
def init_anthropic_instrumentor():
    AnthropicInstrumentor().instrument()


langfuse = init_langfuse()
init_anthropic_instrumentor()
openai_client = OpenAI()
anthropic_client = Anthropic()


def _stream_openai(messages, model):
    stream = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        stream_options={"include_usage": True},
    )
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            yield delta


def _stream_anthropic(messages, model):
    stream = anthropic_client.messages.create(
        model=model,
        max_tokens=1024,
        messages=messages,
        stream=True,
    )
    for event in stream:
        if event.type == "content_block_delta":
            yield event.delta.text


@observe()
def stream_reply(messages, model, trace_holder):
    trace_holder.append(langfuse.get_current_trace_id())
    clean = [{"role": m["role"], "content": m["content"]} for m in messages]
    if MODELS[model] == "openai":
        yield from _stream_openai(clean, model)
    else:
        yield from _stream_anthropic(clean, model)


st.title("Streamlit × Langfuse Demo")

if "messages" not in st.session_state:
    st.session_state.messages = []

if "session_id" not in st.session_state:
    st.session_state.session_id = str(uuid.uuid4())

if "user_id" not in st.session_state:
    st.session_state.user_id = "anonymous"

if "model" not in st.session_state:
    st.session_state.model = "gpt-4o-mini"

with st.sidebar:
    name = st.text_input(
        "Your name",
        value="" if st.session_state.user_id == "anonymous" else st.session_state.user_id,
    )
    st.session_state.user_id = name.strip() or "anonymous"

    st.session_state.model = st.selectbox(
        "Model",
        list(MODELS.keys()),
        index=list(MODELS.keys()).index(st.session_state.model),
    )

    if st.button("New conversation"):
        st.session_state.messages = []
        st.session_state.session_id = str(uuid.uuid4())
        st.rerun()

if "scored_traces" not in st.session_state:
    st.session_state.scored_traces = set()


def score_trace(trace_id: str, value: str):
    langfuse.create_score(
        trace_id=trace_id,
        name="user-feedback",
        value=value,
        data_type="CATEGORICAL",
    )
    langfuse.flush()
    st.session_state.scored_traces.add(trace_id)


for idx, msg in enumerate(st.session_state.messages):
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg["role"] == "assistant" and msg.get("trace_id"):
            trace_id = msg["trace_id"]
            if trace_id in st.session_state.scored_traces:
                st.caption("Thanks for the feedback!")
            else:
                up, down, _ = st.columns([1, 1, 10])
                if up.button("👍", key=f"up_{idx}"):
                    score_trace(trace_id, "positive")
                    st.rerun()
                if down.button("👎", key=f"down_{idx}"):
                    score_trace(trace_id, "negative")
                    st.rerun()

if prompt := st.chat_input("Say something"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        trace_holder = []
        with propagate_attributes(
            session_id=st.session_state.session_id,
            user_id=st.session_state.user_id,
            tags=[MODELS[st.session_state.model]],
        ):
            reply = st.write_stream(
                stream_reply(
                    st.session_state.messages,
                    st.session_state.model,
                    trace_holder,
                )
            )
        trace_id = trace_holder[0] if trace_holder else None
        st.session_state.messages.append(
            {"role": "assistant", "content": reply, "trace_id": trace_id}
        )
        st.rerun()

## Explore data in Langfuse

After chatting with the app for a bit, open your project in Langfuse. You will see traces, sessions, user profiles, and feedback scores populated end-to-end.

Each trace captures the full conversation turn — the `stream_reply` wrapper span, the underlying provider call (`openai.chat` or `anthropic.chat`), session id, user id, provider tag, and any feedback score attached via the 👍 / 👎 buttons. See the [Tracing docs](https://langfuse.com/docs/tracing) for a walkthrough of the trace view, or browse [example traces in the Langfuse public demo project](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4).

The [Sessions](https://langfuse.com/docs/tracing-features/sessions) view groups every trace from the same browser tab, making it easy to replay a full conversation across model switches.

![Langfuse Sessions view showing a multi-turn chat session grouped by session id](/images/docs/session.png)

The [Users](https://langfuse.com/docs/tracing-features/users) view aggregates usage across everyone who typed a name into the sidebar.

![Langfuse Users view listing session, trace, token, and cost aggregates per user id](/images/docs/users-list.png)

If you have any questions or feedback, please join the [Langfuse Discord](https://langfuse.com/discord) or create a new thread on [GitHub Discussions](https://langfuse.com/gh-support).

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more.mdx" -->